In [3]:
import os
import glob
import warnings
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix
from matplotlib.backends.backend_pdf import PdfPages

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

import tensorflow as tf
tf.get_logger().setLevel('ERROR')

# ==========================================
# CẤU HÌNH 
# ==========================================
MODEL_PATH       = r'C:\detection_ddos\notebooks\ddos_v3_final.keras'
ENCODER_PATH     = r'C:\detection_ddos\notebooks\ddos_v3_encoder.pkl'
TEST_FOLDER_PATH = r'C:\detection_ddos\data\CSV-03-11\testing\scaled\scaled (1)\scaled'
PDF_REPORT_NAME  = 'Bao_Cao_Test_DDoS_V3_Standard.pdf'
TIME_STEPS       = 10   # Phải giống lúc train

# ==========================================
# BƯỚC 1: Load encoder
# ==========================================
print("=" * 60)
print("BƯỚC 1 — Load encoder")
print("=" * 60)

encoder = joblib.load(ENCODER_PATH)
known_classes = list(encoder.classes_) 

print(f"✅ Load encoder thành công")
print(f"   Classes đã biết ({len(known_classes)}): {known_classes}")

# ==========================================
# BƯỚC 2: Load model
# ==========================================
print("\n" + "=" * 60)
print("BƯỚC 2 — Load model")
print("=" * 60)

sample_files = glob.glob(os.path.join(TEST_FOLDER_PATH, "*.csv"))
if not sample_files:
    raise FileNotFoundError("Không tìm thấy file CSV trong TEST_FOLDER_PATH")

df_sample = pd.read_csv(sample_files[0])
df_sample.columns = df_sample.columns.str.strip()
drop_cols_sample = [c for c in ['Label', 'Label_Encoded'] if c in df_sample.columns]
n_features = df_sample.drop(columns=drop_cols_sample).shape[1]
print(f"✅ n_features = {n_features}")

model = None
try:
    model = tf.keras.models.load_model(MODEL_PATH)
    print("✅ Load thành công bằng load_model")
except Exception as e1:
    print(f"❌ Cách 1 thất bại: {e1}")

if model is None:
    try:
        model = tf.keras.Sequential([
            tf.keras.layers.Input(shape=(TIME_STEPS, n_features)),
            tf.keras.layers.LSTM(128, activation='tanh', return_sequences=True),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.LSTM(64, activation='tanh'),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.Dropout(0.3),
            tf.keras.layers.Dense(64, activation='relu'),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(len(known_classes), activation='softmax')
        ])
        model.load_weights(MODEL_PATH)
        model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
        print("✅ Load thành công bằng load_weights")
    except Exception as e2:
        print(f"❌ Cách 2 thất bại: {e2}")

if model is None:
    raise RuntimeError("❌ Không load được model — kiểm tra lại MODEL_PATH")

print(f"   Input shape : {model.input_shape}")
print(f"   Output shape: {model.output_shape}")

# ==========================================
# TIỆN ÍCH
# ==========================================
def extract_attack_name(file_name: str) -> str:
    raw   = file_name.replace('.csv', '')
    parts = raw.split('_')
    if 'DrDos' in parts:
        idx = parts.index('DrDos')
        return '_'.join(parts[idx + 1:-1])
    return '_'.join(parts[1:-1])

def create_sequences(data: np.ndarray, labels: np.ndarray, time_steps: int):
    X_seq, y_seq = [], []
    for i in range(len(data) - time_steps):
        X_seq.append(data[i: i + time_steps])
        y_seq.append(labels[i + time_steps - 1])
    return np.array(X_seq), np.array(y_seq)

# ==========================================
# BƯỚC 3: Test từng file
# ==========================================
print("\n" + "=" * 60)
print("BƯỚC 3 — Testing Tiêu chuẩn")
print("=" * 60)

all_files = glob.glob(os.path.join(TEST_FOLDER_PATH, "*.csv"))
print(f"Tìm thấy {len(all_files)} file CSV\n")

file_results = []

with PdfPages(PDF_REPORT_NAME) as pdf:

    for file_path in all_files:
        file_name   = os.path.basename(file_path)
        attack_name = extract_attack_name(file_name)

        # BỎ QUA FILE PORTMAP NHƯ YÊU CẦU CỦA BẠN
        if attack_name.lower() == 'portmap' or attack_name not in known_classes:
            print(f"⏭️ BỎ QUA | {file_name} (Thuộc loại tấn công chưa biết/Portmap)")
            continue

        print(f"\n🚀 XỬ LÝ | {file_name} | attack='{attack_name}'")

        try:
            df = pd.read_csv(file_path)
            df.columns = df.columns.str.strip()

            has_label = 'Label' in df.columns

            if has_label:
                df['Label'] = df['Label'].apply(
                    lambda x: 'Normal' if str(x).strip() in ['0', '0.0', 'BENIGN', 'Normal']
                    else attack_name
                )
                
                # Lọc bỏ các nhãn rác nếu vô tình bị lọt
                df = df[df['Label'].isin(known_classes)]
                y_true_encoded = encoder.transform(df['Label'].values)
            else:
                print(f"   ℹ️ Không có cột Label — chỉ predict, không tính accuracy")

            # --- Features ---
            drop_cols = [c for c in ['Label', 'Label_Encoded'] if c in df.columns]
            X = df.drop(columns=drop_cols).values.astype(np.float32)

            if X.shape[1] != n_features:
                print(f"   ⚠️ features={X.shape[1]} ≠ cần {n_features} → bỏ qua")
                continue

            X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

            # --- Sliding window ---
            if has_label:
                X_seq, y_true_seq = create_sequences(X, y_true_encoded, TIME_STEPS)
            else:
                dummy_labels = np.zeros(len(X))
                X_seq, _     = create_sequences(X, dummy_labels, TIME_STEPS)

            if len(X_seq) == 0:
                print(f"   ⚠️ Quá ít dòng → bỏ qua")
                continue

            # --- Predict Tiêu Chuẩn ---
            probs = model.predict(X_seq, verbose=0)
            y_pred_idx = np.argmax(probs, axis=1)

            # --- Tính accuracy & Vẽ ma trận ---
            if has_label:
                acc = accuracy_score(y_true_seq, y_pred_idx)
                
                file_results.append({
                    'file'    : file_name,
                    'attack'  : attack_name,
                    'accuracy': acc
                })

                # Confusion matrix 
                fig, ax = plt.subplots(figsize=(10, 8))
                
                labels_used = sorted(set(y_true_seq) | set(y_pred_idx))
                names_used  = [known_classes[i] for i in labels_used]
                
                cm = confusion_matrix(y_true_seq, y_pred_idx, labels=labels_used)
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                            xticklabels=names_used, yticklabels=names_used, ax=ax)
                
                ax.set_title(f"Báo cáo Test file: {file_name}\nĐộ chính xác (Accuracy): {acc*100:.2f}%", fontsize=14, pad=15)
                ax.set_ylabel('Thực tế (True Label)', fontsize=12)
                ax.set_xlabel('Dự đoán (Predicted Label)', fontsize=12)
                plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

                plt.tight_layout()
                pdf.savefig(fig, bbox_inches='tight')
                plt.close()

                print(f"   ✅ Hoàn tất! Acc = {acc*100:.2f}%")

        except Exception as e:
            print(f"❌ Lỗi file {file_name}: {e}")

# ==========================================
# BƯỚC 4: Tổng hợp
# ==========================================
print("\n" + "=" * 60)
print("BƯỚC 4 — Tổng hợp kết quả")
print("=" * 60)

if not file_results:
    print("❌ Không có kết quả — kiểm tra lại quá trình chạy")
else:
    df_results = pd.DataFrame(file_results)

    print("\n📋 KẾT QUẢ TỪNG FILE:")
    print(f"{'File':<45} {'Attack Type':<15} {'Accuracy':>15}")
    print("-" * 77)
    for _, row in df_results.iterrows():
        print(f"{row['file']:<45} {row['attack']:<15} {row['accuracy']*100:>14.2f}%")

    mean_acc = df_results['accuracy'].mean()
    print(f"\n✅ TRUNG BÌNH TOÀN BỘ (Mean Accuracy): {mean_acc*100:.2f}%")
    print(f"   PDF báo cáo được lưu tại: {PDF_REPORT_NAME}")

BƯỚC 1 — Load encoder
✅ Load encoder thành công
   Classes đã biết (12): ['DNS', 'LDAP', 'MSSQL', 'NTP', 'NetBIOS', 'Normal', 'SNMP', 'SSDP', 'Syn', 'TFTP', 'UDP', 'UDPLag']

BƯỚC 2 — Load model
✅ n_features = 25
❌ Cách 1 thất bại: <class 'keras.src.models.sequential.Sequential'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras', 'class_name': 'Sequential', 'config': {'name': 'sequential_1', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None, 'shared_object_id': 3210478356864}, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_shape': [None, 10, 25], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_layer_1', 'optional': False}, 'registered_name': None}, {'module':